# 📖 Notebook 3: Market Data Streaming

Robinhood shows **live stock prices** that update in real time. This notebook explores
how an exchange trade feed flows through Kafka, gets processed, and is pushed to users
via Redis pub/sub.

## Learning Objectives

By the end of this notebook you will understand:
- How an exchange **trade feed** delivers price updates
- How **Kafka** acts as a durable buffer for high-throughput trade events
- How **Redis pub/sub** fans out price updates to connected servers
- Why **Server-Sent Events (SSE)** are preferred over polling or WebSockets for this use case
- How the full pipeline fits together: Exchange → Kafka → Processor → Redis → Client

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/robinhood
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `robinhood_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import random
import threading
from datetime import datetime, timezone
from kafka import KafkaProducer, KafkaConsumer
from kafka.errors import NoBrokersAvailable

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "robinhood_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

KAFKA_BROKER = "localhost:9094"

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test all three connections
try:
    conn = get_db()
    conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

try:
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BROKER,
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    print("✅ Kafka connected")
    producer.close()
except NoBrokersAvailable:
    print("❌ Kafka failed — run: docker-compose up -d")

## 📚 The Architecture

Here's how live prices flow from the exchange to the user's screen:

```
  Exchange          Our Backend                              User
  ────────         ────────────                             ────
                                                              
  Trade Feed ─────► Kafka ─────► Price       Redis ──────► Symbol
  (external)        topic:       Processor    Pub/Sub       Service ───► SSE ──► App
                    trades       (consumer)   (fan-out)     (server)
                                    │
                                    └──► Postgres
                                         (price_history)
```

### Why This Pipeline?

| Component | Role | Why Not Skip It? |
|-----------|------|------------------|
| **Kafka** | Durable buffer for trade events | Absorbs bursts; replays on failure |
| **Price Processor** | Consumes trades, updates DB + cache | Single writer avoids conflicts |
| **Redis Pub/Sub** | Broadcasts to all symbol servers | Decouples processors from servers |
| **SSE** | Pushes to user's browser/app | No polling; server-initiated updates |

### Why SSE Instead of WebSockets?

- Price updates are **one-directional** (server → client)
- SSE works over standard HTTP (simpler load balancer config)
- Automatic reconnection built into the protocol
- WebSockets would be overkill — the client doesn't send price data back

## 1️⃣ Simulating the Exchange Trade Feed with Kafka

In the real world, the exchange pushes trades to us via a feed.  
We'll simulate this by producing trade events into a Kafka topic.

In [ ]:
# Load our symbols and their current prices
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT ticker, last_price_cents FROM symbols ORDER BY ticker")
symbols = {row['ticker']: row['last_price_cents'] for row in cur.fetchall()}
conn.close()

print("📊 Symbols loaded:")
for ticker, price in symbols.items():
    print(f"  {ticker:<6} ${price/100:.2f}")

In [ ]:
def simulate_exchange_feed(num_trades: int = 20, delay: float = 0.3):
    """
    Simulate an exchange trade feed by producing random trades into Kafka.
    Each trade slightly moves the price up or down.
    """
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BROKER,
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )

    print(f"📡 Producing {num_trades} trades to Kafka topic 'trades'...")
    print()

    for i in range(num_trades):
        ticker = random.choice(list(symbols.keys()))
        current_price = symbols[ticker]

        # Random price movement: ±0.5%
        change_pct = random.uniform(-0.005, 0.005)
        new_price = int(current_price * (1 + change_pct))
        symbols[ticker] = new_price  # update our local tracker

        trade = {
            "ticker": ticker,
            "price_cents": new_price,
            "quantity": random.randint(1, 100),
            "timestamp": datetime.now(timezone.utc).isoformat()
        }

        producer.send("trades", value=trade)

        direction = "📈" if change_pct > 0 else "📉"
        print(f"  {direction} {ticker:<6} ${new_price/100:.2f}  "
              f"({change_pct:+.2%})  ×{trade['quantity']} shares")

        time.sleep(delay)

    producer.flush()
    producer.close()
    print(f"\n✅ {num_trades} trades published to Kafka")


simulate_exchange_feed(num_trades=15, delay=0.2)

## 2️⃣ Price Processor: Consuming from Kafka

The **price processor** reads trades from Kafka and does two things:
1. Updates the `last_price_cents` in Postgres (and stores price history)
2. Publishes the price update to Redis pub/sub so connected servers get notified

In [ ]:
def run_price_processor(max_messages: int = 15, timeout_ms: int = 5000):
    """
    Consume trades from Kafka, update Postgres, publish to Redis pub/sub.
    
    In production this runs continuously as a service.
    Here we process a fixed number for demonstration.
    """
    consumer = KafkaConsumer(
        "trades",
        bootstrap_servers=KAFKA_BROKER,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='earliest',
        consumer_timeout_ms=timeout_ms,
        group_id=f'price-processor-{int(time.time())}'
    )

    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    processed = 0
    print("⚙️  Price Processor running...")
    print()

    for message in consumer:
        trade = message.value
        ticker = trade['ticker']
        price = trade['price_cents']

        # 1. Update latest price in Postgres
        cur.execute("""
            UPDATE symbols SET last_price_cents = %s, updated_at = NOW()
            WHERE ticker = %s
            RETURNING id
        """, (price, ticker))
        result = cur.fetchone()

        if result:
            symbol_id = result[0]
            # Store in price history
            cur.execute("""
                INSERT INTO price_history (symbol_id, price_cents) VALUES (%s, %s)
            """, (symbol_id, price))

        # 2. Also cache latest price in Redis for fast lookups
        r.set(f"price:{ticker}", price)

        # 3. Publish to Redis pub/sub channel
        update_msg = json.dumps({
            "ticker": ticker,
            "price_cents": price,
            "timestamp": trade['timestamp']
        })
        r.publish(f"price:{ticker}", update_msg)

        processed += 1
        print(f"  ✅ [{processed}] {ticker} → ${price/100:.2f}  "
              f"(DB ✓, Cache ✓, Pub/Sub ✓)")

        if processed >= max_messages:
            break

    consumer.close()
    conn.close()
    print(f"\n⚙️  Processed {processed} trade(s)")


run_price_processor()

## 3️⃣ Redis Pub/Sub: Fan-Out to Symbol Servers

In production, many **symbol service** servers maintain SSE connections to users.
Each server subscribes to Redis pub/sub channels for the symbols its connected users
care about.

```
  Price Processor ──publish──► Redis Pub/Sub
                                    │
                    ┌───────────────┼───────────────┐
                    ▼               ▼               ▼
              Symbol Server 1  Server 2        Server 3
              (AAPL, META)    (AAPL, TSLA)    (NVDA, META)
                    │               │               │
                    ▼               ▼               ▼
                Users A,B       Users C,D       Users E,F
```

Redis pub/sub is perfect here because:
- It's **fire-and-forget** — no message persistence needed (prices are ephemeral)
- It's **fast** — sub-millisecond delivery
- Servers subscribe only to channels they need

In [ ]:
# Simulate a symbol server subscribing to price updates

received_updates = []  # collect updates for display
stop_flag = threading.Event()

def symbol_server(subscribed_tickers: list):
    """
    Simulates a symbol service server that listens to Redis pub/sub
    for price updates and would push them to connected users via SSE.
    """
    r = get_redis()
    pubsub = r.pubsub()

    # Subscribe to channels for each ticker
    channels = [f"price:{t}" for t in subscribed_tickers]
    pubsub.subscribe(*channels)
    print(f"🖥️  Symbol server subscribed to: {', '.join(subscribed_tickers)}")

    for message in pubsub.listen():
        if stop_flag.is_set():
            break
        if message['type'] == 'message':
            data = json.loads(message['data'])
            received_updates.append(data)

    pubsub.unsubscribe()
    pubsub.close()


# Start the "server" in a background thread
stop_flag.clear()
received_updates.clear()
server_thread = threading.Thread(
    target=symbol_server,
    args=(["AAPL", "TSLA", "NVDA"],)
)
server_thread.daemon = True
server_thread.start()
time.sleep(0.5)  # let it subscribe

print("Server is listening... now let's publish some price updates.")

In [ ]:
# Publish some price updates (simulating what the price processor does)

r = get_redis()

test_updates = [
    {"ticker": "AAPL",  "price_cents": 19200, "timestamp": datetime.now(timezone.utc).isoformat()},
    {"ticker": "TSLA",  "price_cents": 24800, "timestamp": datetime.now(timezone.utc).isoformat()},
    {"ticker": "META",  "price_cents": 52500, "timestamp": datetime.now(timezone.utc).isoformat()},  # not subscribed!
    {"ticker": "NVDA",  "price_cents": 89000, "timestamp": datetime.now(timezone.utc).isoformat()},
    {"ticker": "AAPL",  "price_cents": 19250, "timestamp": datetime.now(timezone.utc).isoformat()},
]

for update in test_updates:
    r.publish(f"price:{update['ticker']}", json.dumps(update))
    print(f"  📡 Published: {update['ticker']} → ${update['price_cents']/100:.2f}")
    time.sleep(0.2)

time.sleep(0.5)  # let messages arrive

# Stop the server
stop_flag.set()
# Unblock the listener by publishing a dummy message
r.publish("price:AAPL", json.dumps({"ticker": "AAPL", "price_cents": 0, "timestamp": ""}))
server_thread.join(timeout=2)

# Show what the server received
# Filter out the dummy unblock message
real_updates = [u for u in received_updates if u['price_cents'] > 0]
print(f"\n🖥️  Symbol server received {len(real_updates)} updates:")
for u in real_updates:
    print(f"  {u['ticker']:<6} ${u['price_cents']/100:.2f}")

print()
print("💡 Notice: META update was NOT received — the server wasn't subscribed to it!")
print("   This is the power of pub/sub: servers only get what they need.")

## 4️⃣ Full Pipeline Demo

Let's run the entire pipeline end-to-end:
1. Exchange produces trades → Kafka
2. Price processor reads Kafka → updates DB + publishes to Redis
3. Symbol server receives Redis pub/sub updates

We'll run each piece in a thread to simulate a real system.

In [ ]:
# Full end-to-end pipeline

pipeline_updates = []
pipeline_stop = threading.Event()


def pipeline_subscriber(tickers):
    """Symbol server collecting updates."""
    r = get_redis()
    ps = r.pubsub()
    ps.subscribe(*[f"price:{t}" for t in tickers])
    for msg in ps.listen():
        if pipeline_stop.is_set():
            break
        if msg['type'] == 'message':
            data = json.loads(msg['data'])
            if data.get('price_cents', 0) > 0:
                pipeline_updates.append({
                    **data,
                    "received_at": time.time()
                })
    ps.close()


def pipeline_processor():
    """Price processor consuming from Kafka."""
    consumer = KafkaConsumer(
        "trades",
        bootstrap_servers=KAFKA_BROKER,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        consumer_timeout_ms=8000,
        group_id=f'pipeline-demo-{int(time.time())}'
    )
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    for msg in consumer:
        trade = msg.value
        # Update DB
        cur.execute("UPDATE symbols SET last_price_cents = %s WHERE ticker = %s RETURNING id",
                    (trade['price_cents'], trade['ticker']))
        result = cur.fetchone()
        if result:
            cur.execute("INSERT INTO price_history (symbol_id, price_cents) VALUES (%s, %s)",
                        (result[0], trade['price_cents']))
        # Cache + pub/sub
        r.set(f"price:{trade['ticker']}", trade['price_cents'])
        r.publish(f"price:{trade['ticker']}", json.dumps(trade))

    consumer.close()
    conn.close()


# Start subscriber (all symbols)
pipeline_stop.clear()
pipeline_updates.clear()
all_tickers = list(symbols.keys())
sub_thread = threading.Thread(target=pipeline_subscriber, args=(all_tickers,))
sub_thread.daemon = True
sub_thread.start()
time.sleep(0.5)

# Start processor
proc_thread = threading.Thread(target=pipeline_processor)
proc_thread.daemon = True
proc_thread.start()
time.sleep(1)

# Produce trades
print("🚀 Full Pipeline: Exchange → Kafka → Processor → Redis → Subscriber")
print("=" * 65)
simulate_exchange_feed(num_trades=10, delay=0.3)

# Wait for processing
time.sleep(3)
pipeline_stop.set()

# Unblock subscriber
r_temp = get_redis()
r_temp.publish("price:AAPL", json.dumps({"ticker": "AAPL", "price_cents": 0, "timestamp": ""}))
sub_thread.join(timeout=2)
proc_thread.join(timeout=2)

print(f"\n📊 Subscriber received {len(pipeline_updates)} price updates")
print()

# Show unique tickers updated
ticker_counts = {}
for u in pipeline_updates:
    ticker_counts[u['ticker']] = ticker_counts.get(u['ticker'], 0) + 1

print("Updates per ticker:")
for ticker, count in sorted(ticker_counts.items()):
    print(f"  {ticker:<6} {count} update(s)")

## 5️⃣ Fast Price Lookups from Redis Cache

While pub/sub delivers real-time updates, sometimes a user opens the app and needs
the *current* price immediately (before the next update arrives).

We store the latest price in Redis as a simple key-value for O(1) lookups.

In [ ]:
r = get_redis()

def get_live_prices(tickers: list) -> dict:
    """Fetch latest prices from Redis cache (sub-millisecond)."""
    pipe = r.pipeline()
    for ticker in tickers:
        pipe.get(f"price:{ticker}")
    results = pipe.execute()

    prices = {}
    for ticker, val in zip(tickers, results):
        if val:
            prices[ticker] = int(val)
    return prices


# Measure latency
start = time.time()
prices = get_live_prices(all_tickers)
elapsed_ms = (time.time() - start) * 1000

print(f"⚡ Fetched {len(prices)} prices in {elapsed_ms:.2f} ms")
print()
for ticker, price in sorted(prices.items()):
    print(f"  {ticker:<6} ${price/100:.2f}")

print()
print("💡 Redis pipeline fetches ALL prices in a single round-trip.")
print("   This is what powers the initial page load before SSE kicks in.")

## Bad Practice -> Best Practice: Polling vs Pub/Sub

A simple way to get prices to users would be to have the browser poll an HTTP endpoint every few seconds: `GET /price/AAPL` every 3 s. That's easy to build... and a disaster at scale.

- **Bad (polling)**: every client makes one request per interval whether or not the price changed. 1 million users polling every 3 s = **333 k req/s** of mostly-wasted traffic, and the user still sees updates up to 3 s late.
- **Good (pub/sub + SSE)**: the server pushes *only when the price actually changes*. One message per change, fanned out to just the subscribers who want that symbol. Sub-second latency, and near-zero traffic when the market is quiet.

Let's measure the two side by side.


In [ ]:
import statistics

r = get_redis()

# Load latest prices into Redis so the polling client has something to GET
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT ticker, last_price_cents FROM symbols")
for row in cur.fetchall():
    r.set(f"price:{row['ticker']}", row['last_price_cents'])
conn.close()

TICKER = "AAPL"
POLL_INTERVAL_S = 0.25     # the client polls 4x / second
DURATION_S = 2.0

# -------- BAD: polling --------
bad_requests = 0
bad_latencies_ms = []
# We'll publish a single price change in the middle of the window and see how long the poller takes to notice.
start = time.perf_counter()
change_time = None
new_price = 99999
saw_at = None
last_seen = int(r.get(f"price:{TICKER}"))
# Kick the change at t=1s
def schedule_change():
    global change_time
    time.sleep(1.0)
    change_time = time.perf_counter()
    r.set(f"price:{TICKER}", new_price)

threading.Thread(target=schedule_change, daemon=True).start()

while time.perf_counter() - start < DURATION_S:
    bad_requests += 1
    cur_price = int(r.get(f"price:{TICKER}"))
    if cur_price != last_seen and saw_at is None:
        saw_at = time.perf_counter()
    time.sleep(POLL_INTERVAL_S)

bad_latency_ms = (saw_at - change_time) * 1000 if saw_at and change_time else float("nan")
print(f"BAD  polling:  {bad_requests} requests in {DURATION_S}s, latency to notice change = {bad_latency_ms:.0f} ms")

# Reset
r.set(f"price:{TICKER}", last_seen)

# -------- GOOD: pub/sub --------
good_messages = 0
pubsub_latency_ms = None
stop = threading.Event()

def subscriber():
    global good_messages, pubsub_latency_ms
    rr = get_redis()
    ps = rr.pubsub()
    ps.subscribe(f"price:{TICKER}")
    for msg in ps.listen():
        if stop.is_set():
            break
        if msg['type'] == 'message':
            good_messages += 1
            if pubsub_latency_ms is None:
                pubsub_latency_ms = (time.perf_counter() - publish_time) * 1000
    ps.close()

t = threading.Thread(target=subscriber, daemon=True)
t.start()
time.sleep(0.3)  # let it subscribe

# Publish a single price change -- the only "traffic" that happens
publish_time = time.perf_counter()
r.publish(f"price:{TICKER}", json.dumps({"ticker": TICKER, "price_cents": new_price}))

time.sleep(0.5)
stop.set()
# Unblock the listener
r.publish(f"price:{TICKER}", json.dumps({"ticker": TICKER, "price_cents": last_seen}))
t.join(timeout=2)

print(f"GOOD pub/sub:  {good_messages} message(s) delivered, latency to notice change = {pubsub_latency_ms:.1f} ms")
print()
print("Tip: polling sends traffic even when nothing happens, and its worst-case latency")
print("     is the poll interval. Pub/sub sends 1 message per real change and arrives in milliseconds.")


## 📐 Scaling Considerations

In a real system with millions of users:

### 1. Kafka Partitioning
Partition the `trades` topic by symbol ticker. This ensures all trades for AAPL
go to the same partition → processed in order → no race conditions on price updates.

### 2. Redis Pub/Sub Channel Design
One channel per symbol (e.g., `price:AAPL`). Servers subscribe only to channels
that their connected users care about. If no user on a server watches AAPL,
that server doesn't subscribe to `price:AAPL`.

### 3. Throttling Updates to Clients
If AAPL trades 1000 times per second, we don't need to push every single trade.
The price processor can **batch** updates — e.g., publish at most once per 100ms
per symbol, sending only the latest price.

### 4. Sticky Sessions for SSE
SSE connections are long-lived. The load balancer must use **sticky sessions**
so a user's SSE connection always reaches the same server.

## 🧹 Cleanup

In [ ]:
# Clean up Redis price keys
r = get_redis()
keys = r.keys("price:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

1. **Kafka** buffers the exchange trade feed — durable, replayable, handles bursts.
2. **Price Processor** consumes from Kafka and writes to DB + Redis.
3. **Redis Pub/Sub** fans out price updates to only the servers that need them.
4. **Redis cache** stores latest prices for instant lookups on page load.
5. **SSE** pushes updates to clients — simpler than WebSockets for one-way data.
6. **Don't poll.** Pub/sub sends traffic only when prices actually change and delivers in milliseconds.

### For System Design Interviews

- Draw the full pipeline: Exchange → Kafka → Processor → Redis → SSE → Client
- Explain why SSE over WebSockets (unidirectional, simpler)
- Mention Kafka partitioning by symbol for ordering guarantees
- Discuss throttling to avoid overwhelming clients with rapid updates
- Note Redis pub/sub is fire-and-forget (no persistence) — fine for ephemeral price data
- Sticky sessions needed for SSE at the load balancer

### The Big Picture

Across all three notebooks, we've covered the core of a brokerage system:

| Notebook | What We Built |
|----------|---------------|
| 1. Order Matching Engine | Order types, order book, lifecycle, consistency |
| 2. Portfolio Tracking | Positions, P&L, transactions, caching |
| 3. Market Data Streaming | Kafka pipeline, Redis pub/sub, live prices |

Together these form the backbone of a system like Robinhood! 🎉